In [26]:
import pandas as pd

from tqdm import tqdm
from rapidfuzz import process, fuzz
import re
import unicodedata
import asyncio
import aiohttp
from utils.fetch_data import fetch_soil_async, fetch_weather_async

In [27]:
def normalize_text(x):
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x


def fuzzy_match_district(df, centroid, threshold=85):
    # Normalize both sides
    df["District_norm"] = df["District"].apply(normalize_text)
    centroid["District_norm"] = centroid["District"].apply(normalize_text)
    centroid["State_norm"] = centroid["State"].apply(normalize_text)
    df["State_norm"] = df["State"].apply(normalize_text)

    # Drop duplicates in centroid to prevent merge explosion
    centroid = centroid.drop_duplicates(subset=["State_norm", "District_norm"])

    # Build fast lookup per state
    state_to_districts = (
        centroid.groupby("State_norm")["District_norm"].apply(list).to_dict()
    )

    def match_row(row):
        dist = row["District_norm"]
        state = row["State_norm"]

        if state in state_to_districts and dist in state_to_districts[state]:
            return dist

        # Try fuzzy match in same state first
        if state in state_to_districts:
            match, score, _ = process.extractOne(
                dist, state_to_districts[state], scorer=fuzz.WRatio
            )  # type: ignore
            if score >= threshold:
                return match

        # If still not found, fuzzy match across all districts
        all_dists = centroid["District_norm"].tolist()
        match, score, _ = process.extractOne(dist, all_dists, scorer=fuzz.WRatio)  # type: ignore
        if score >= threshold:
            return match

        return dist

    df["District_norm"] = df.apply(match_row, axis=1)

    merged = df.merge(
        centroid[["State_norm", "District_norm", "Latitude", "Longitude"]].rename(
            columns={
                "Latitude": "latitude",
                "Longitude": "longitude",
            }
        ),
        on=["State_norm", "District_norm"],
        how="left",
    )

    return merged

In [28]:
import os

df_new = pd.read_csv("data/converted_crop_data.csv")
df_new["Start_Year"] = df_new["Year"].apply(lambda x: int(x.split("-")[0]))

if os.path.exists("data/enriched_crop_data.csv"):
    df_existing = pd.read_csv("data/enriched_crop_data.csv")
    # Create a key for merging/filtering
    keys = ["State", "District", "Season", "Year", "Crop"]

    # Perform an anti-join to find rows in df_new that are not in df_existing
    merged_check = df_new.merge(df_existing[keys], on=keys, how="left", indicator=True)
    df_to_process = merged_check[merged_check["_merge"] == "left_only"].drop(
        columns=["_merge"]
    )
else:
    df_existing = pd.DataFrame()
    df_to_process = df_new

print(f"Total rows: {len(df_new)}")
print(f"Existing rows: {len(df_existing)}")
print(f"Rows to process: {len(df_to_process)}")

if not df_to_process.empty:
    centroid = pd.read_csv("data/centroid.csv")
    df_merged = fuzzy_match_district(df_to_process, centroid)
    df_merged.dropna(subset=["latitude", "longitude"], inplace=True)
    df_merged.info()
else:
    df_merged = pd.DataFrame()
    print("No new data to process.")

Total rows: 143418
Existing rows: 58953
Rows to process: 84465
<class 'pandas.core.frame.DataFrame'>
Index: 71990 entries, 0 to 83196
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   State          71990 non-null  object 
 1   District       71990 non-null  object 
 2   Season         71990 non-null  object 
 3   Year           71990 non-null  object 
 4   Crop           71990 non-null  object 
 5   Area_Ha        71990 non-null  float64
 6   Yield_QHa      71990 non-null  float64
 7   Start_Year     71990 non-null  int64  
 8   District_norm  71990 non-null  object 
 9   State_norm     71990 non-null  object 
 10  latitude       71990 non-null  float64
 11  longitude      71990 non-null  float64
dtypes: float64(4), int64(1), object(7)
memory usage: 7.1+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 71990 entries, 0 to 83196
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  --

In [29]:
async def process_soil_batch(unique_df):
    results = []
    async with aiohttp.ClientSession() as session:
        tasks = []
        for _, row in unique_df.iterrows():
            task = fetch_soil_async(session, row["latitude"], row["longitude"])
            tasks.append(task)

        print(f"Fetching soil data for {len(tasks)} locations...")

        # Wrap tasks with progress bar
        pbar = tqdm(total=len(tasks), desc="Soil Data")

        async def wrap_task(t):
            res = await t
            pbar.update(1)
            return res

        wrapped_tasks = [wrap_task(t) for t in tasks]
        api_results = await asyncio.gather(*wrapped_tasks)
        pbar.close()

        # Merge back with metadata
        for i, res in enumerate(api_results):
            if res:
                item = res.copy()
                item["State"] = unique_df.iloc[i]["State"]
                item["District"] = unique_df.iloc[i]["District"]
                results.append(item)

    return pd.DataFrame(results)


async def process_weather_batch(unique_df):
    results = []
    async with aiohttp.ClientSession() as session:
        tasks = []
        for _, row in unique_df.iterrows():
            task = fetch_weather_async(
                session,
                row["latitude"],
                row["longitude"],
                row["Start_Year"],
                row["Season"],
            )
            tasks.append(task)

        print(f"Fetching weather data for {len(tasks)} combinations...")

        # Wrap tasks with progress bar
        pbar = tqdm(total=len(tasks), desc="Weather Data")

        async def wrap_task(t):
            res = await t
            pbar.update(1)
            return res

        wrapped_tasks = [wrap_task(t) for t in tasks]
        api_results = await asyncio.gather(*wrapped_tasks)
        pbar.close()

        for i, res in enumerate(api_results):
            if res:
                item = res.copy()
                item["State"] = unique_df.iloc[i]["State"]
                item["District"] = unique_df.iloc[i]["District"]
                item["Season"] = unique_df.iloc[i]["Season"]
                item["Start_Year"] = unique_df.iloc[i]["Start_Year"]
                results.append(item)

    return pd.DataFrame(results)


async def append_data_async(df_merged):
    # 1. Unique Soil Requests
    soil_keys = df_merged[
        ["State", "District", "latitude", "longitude"]
    ].drop_duplicates()

    # Run async loop for soil
    soil_df = await process_soil_batch(soil_keys)

    # Merge Soil
    if not soil_df.empty:
        df_merged = df_merged.merge(soil_df, on=["State", "District"], how="left")

    # 2. Unique Weather Requests
    weather_keys = df_merged[
        ["State", "District", "latitude", "longitude", "Season", "Start_Year"]
    ].drop_duplicates()

    # Run async loop for weather
    weather_df = await process_weather_batch(weather_keys)

    # Merge Weather
    if not weather_df.empty:
        df_merged = df_merged.merge(
            weather_df, on=["State", "District", "Season", "Start_Year"], how="left"
        )

    return df_merged

In [30]:
if not df_merged.empty:
    # Use the new async function
    df_processed = await append_data_async(df_merged)

    if not df_existing.empty:
        df_final = pd.concat([df_existing, df_processed], ignore_index=True)
    else:
        df_final = df_processed
else:
    df_final = df_existing

# Drop duplicates based on key columns
if not df_final.empty:
    df_final.drop_duplicates(
        subset=["State", "District", "Season", "Year", "Crop"], inplace=True
    )

print("Processing complete.")
df_final.head()

Fetching soil data for 325 locations...


Soil Data: 100%|██████████| 325/325 [06:20<00:00,  1.17s/it]



Fetching weather data for 8547 combinations...


Weather Data: 100%|██████████| 8547/8547 [1:54:51<00:00,  1.24it/s]  



Processing complete.


,State,District,Season,Year,Crop,Area_Ha,Yield_QHa,Start_Year,District_norm,State_norm,...,longitude,soil_ph,soil_oc,clay_pct,sand_pct,cec_cmol,avg_temp,humidity_avg,rain_total,solar_avg
0,Chandigarh,Chandigarh,Whole Year,2015 - 2016,Potato,10.0,155.0,2015,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,23.899342,50.669836,1130.12,18.550904
1,Chandigarh,Chandigarh,Kharif,2015 - 2016,Rice,20.0,51.5,2015,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,29.421639,62.886967,840.64,21.009426
2,Chandigarh,Chandigarh,Kharif,2016 - 2017,Rice,8.0,52.5,2016,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,30.185328,63.451721,712.46,20.838361
3,Chandigarh,Chandigarh,Kharif,2017 - 2018,Rice,10.0,55.0,2017,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,29.200328,67.663607,991.37,20.418279
4,Chandigarh,Chandigarh,Kharif,2018 - 2019,Rice,10.0,53.0,2018,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,28.788443,72.857705,1270.35,18.903033


In [31]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130943 entries, 0 to 130942
Data columns (total 21 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   State          130943 non-null  object 
 1   District       130943 non-null  object 
 2   Season         130943 non-null  object 
 3   Year           130943 non-null  object 
 4   Crop           130943 non-null  object 
 5   Area_Ha        130943 non-null  float64
 6   Yield_QHa      130943 non-null  float64
 7   Start_Year     130943 non-null  int64  
 8   District_norm  130943 non-null  object 
 9   State_norm     130943 non-null  object 
 10  latitude       130943 non-null  float64
 11  longitude      130943 non-null  float64
 12  soil_ph        117817 non-null  float64
 13  soil_oc        117817 non-null  float64
 14  clay_pct       117817 non-null  float64
 15  sand_pct       117817 non-null  float64
 16  cec_cmol       117817 non-null  float64
 17  avg_temp       120965 non-nul

In [33]:
df_final.to_csv("data/enriched_crop_data_v1.csv", index=False)